# EXP_009d1: Attractor Dominance, Basin Mapping & Pathway Analysis

## Depends On
**Stage 0 (Determinism Check) must complete and produce repeatable terminal basins before running this notebook.**

## Hypotheses Under Test

The hypotheses below were pre-registered for the Stage 1 sweep. The post-hoc calibrated assessment of each (using the supervisor's structural-vs-predictive framing and the embedding-neighbourhood evidence) appears in the project README and `docs/JOURNEY_MAP.md`.

### H1: Attractor Dominance
The `prolet` attractor is the dominant basin of GPT-2 Small's weight geometry. It should capture the majority of a diverse prompt set, regardless of input register, topic, or complexity.

### H2: Secondary Basin Existence
The `Divine` attractor is a genuine secondary basin, not a one-off artefact of a single prompt. Other prompts with similar syntactic properties should also route to `Divine` (or to other previously unseen basins).

### H3: Dissolution Pathway Structure
The intermediate tokens observed during dissolution (e.g., `Femminus Fem`) reflect the statistical topology of the training corpus, not random noise. Different input types may trace different but internally coherent pathways to the same terminal attractor.

---


In [10]:
# ============================================================
# STEP 0: DEPENDENCIES
# ============================================================
import sys
!{sys.executable} -m pip install kaleido -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ============================================================
# STEP 1: SETUP
# ============================================================
import torch
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Architecture: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")
print(f"Running on: {device}")

# Output directory for all saved artifacts (relative path — keeps output stable across machines)
OUTPUT_DIR = "output_stage1"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [12]:
# ============================================================
# STEP 2: CONFIGURATION — 125 Prompts from prompt_library.py
# ============================================================
from prompt_library import (
    PROMPT_LIBRARY, PREDICTIONS, CATEGORY_MAP,
    COMPLEX, NARRATIVE, SIMPLE, CHEMICAL, ACRONYMS, VULGARITY, WILD
)

# Tightened schedule: convergence occurs by ~100
ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

LAYER_START = 0
LAYER_END = model.cfg.n_layers - 1

print(f"Schedule: {ITERATION_SCHEDULE}")
print(f"Room: Layers {LAYER_START} -> {LAYER_END}")
print(f"Total prompts: {len(PROMPT_LIBRARY)}")
print(f"\nBreakdown:")
for cat_name, cat_dict in [
    ("Complex", COMPLEX), ("Narrative", NARRATIVE),
    ("Simple", SIMPLE), ("Chemical", CHEMICAL),
    ("Acronyms", ACRONYMS), ("Vulgarity", VULGARITY),
    ("Wild", WILD)
]:
    print(f"  {cat_name}: {len(cat_dict)} prompts")

# Save config
config_md = f"""# Stage 1 Run Config\n
- **Prompts:** {len(PROMPT_LIBRARY)}\n
- **Schedule:** {ITERATION_SCHEDULE}\n
- **Layers:** {LAYER_START} -> {LAYER_END}\n
- **Device:** {device}\n
"""
with open(os.path.join(OUTPUT_DIR, 'config.md'), 'w', encoding='utf-8') as f:
    f.write(config_md)
print(f"\n[SAVED] {OUTPUT_DIR}/config.md")

Schedule: [0, 2, 3, 5, 10, 20, 50, 100]
Room: Layers 0 -> 11
Total prompts: 125

Breakdown:
  Complex: 25 prompts
  Narrative: 20 prompts
  Simple: 20 prompts
  Chemical: 10 prompts
  Acronyms: 10 prompts
  Vulgarity: 10 prompts
  Wild: 30 prompts

[SAVED] output_stage1/config.md


In [13]:
# ============================================================
# STEP 3: THE CORE ENGINE — Identical to lucier_total_resonance
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions.
    Applies the Final LayerNorm before unembedding for correct decoding."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_total_resonance_loop(model, prompt, layer_start, layer_end, max_iter, schedule):
    """
    TOTAL Lucier Loop: iteratively re-inject the ENTIRE residual stream
    tensor (all token positions) through the layer slice.
    Returns a list of snapshot dicts at each scheduled iteration.
    """
    snapshots = []
    hook_point_read = f"blocks.{layer_end}.hook_resid_post"
    hook_point_write = f"blocks.{layer_start}.hook_resid_pre"
    
    with torch.no_grad():
        _, cache = model.run_with_cache(
            prompt,
            names_filter=lambda n: n == hook_point_read
        )
    
    current_tensor = cache[hook_point_read][0].clone()
    seq_len = current_tensor.shape[0]
    initial_norm = current_tensor.norm().item()
    
    last_vec = current_tensor[-1, :].clone()
    mean_vec = current_tensor.mean(dim=0).clone()
    
    if 0 in schedule:
        top_tokens_last = get_top_tokens(model, last_vec)
        all_pos_tokens = []
        for pos in range(seq_len):
            pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
            all_pos_tokens.append(pos_top[0][0])
        snapshots.append({
            "iteration": 0,
            "tensor": current_tensor.clone().cpu(),
            "last_vector": last_vec.clone().cpu(),
            "mean_vector": mean_vec.clone().cpu(),
            "last_norm": last_vec.norm().item(),
            "mean_norm": mean_vec.norm().item(),
            "tensor_norm": current_tensor.norm().item(),
            "top_tokens": top_tokens_last,
            "all_position_tokens": all_pos_tokens,
            "cosine_sim_last": 1.0,
            "cosine_sim_mean": 1.0,
            "position_similarity": 1.0,
        })
    
    prev_last = last_vec.clone()
    prev_mean = mean_vec.clone()
    
    for i in range(1, max_iter + 1):
        # Normalise to maintain energy level
        current_norm = current_tensor.norm().item()
        if current_norm > 0:
            current_tensor = current_tensor * (initial_norm / current_norm)
        
        inject_tensor = current_tensor.clone()
        
        def injection_hook(resid, hook, tensor=inject_tensor):
            resid[0, :, :] = tensor
            return resid
        
        model.add_hook(hook_point_write, injection_hook)
        try:
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    prompt,
                    names_filter=lambda n: n == hook_point_read
                )
        finally:
            model.reset_hooks()
        
        current_tensor = cache[hook_point_read][0].clone()
        last_vec = current_tensor[-1, :].clone()
        mean_vec = current_tensor.mean(dim=0).clone()
        
        if i in schedule:
            cos_sim_last = torch.nn.functional.cosine_similarity(
                last_vec.unsqueeze(0), prev_last.unsqueeze(0)
            ).item()
            cos_sim_mean = torch.nn.functional.cosine_similarity(
                mean_vec.unsqueeze(0), prev_mean.unsqueeze(0)
            ).item()
            
            pos_norms = current_tensor.norm(dim=1, keepdim=True).clamp(min=1e-8)
            normalized_positions = current_tensor / pos_norms
            pos_sim_matrix = normalized_positions @ normalized_positions.T
            mask = ~torch.eye(seq_len, dtype=torch.bool, device=pos_sim_matrix.device)
            position_similarity = pos_sim_matrix[mask].mean().item()
            
            top_tokens_last = get_top_tokens(model, last_vec)
            all_pos_tokens = []
            for pos in range(seq_len):
                pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
                all_pos_tokens.append(pos_top[0][0])
            
            snapshots.append({
                "iteration": i,
                "tensor": current_tensor.clone().cpu(),
                "last_vector": last_vec.clone().cpu(),
                "mean_vector": mean_vec.clone().cpu(),
                "last_norm": last_vec.norm().item(),
                "mean_norm": mean_vec.norm().item(),
                "tensor_norm": current_tensor.norm().item(),
                "top_tokens": top_tokens_last,
                "all_position_tokens": all_pos_tokens,
                "cosine_sim_last": cos_sim_last,
                "cosine_sim_mean": cos_sim_mean,
                "position_similarity": position_similarity,
            })
            print(f"  iter {i:>3}: top='{top_tokens_last[0][0].strip()}', "
                  f"cos_mean={cos_sim_mean:.4f}, pos_collapse={position_similarity:.4f}")
        
        prev_last = last_vec.clone()
        prev_mean = mean_vec.clone()
    
    return snapshots

print("Engine loaded.")

Engine loaded.


In [ ]:
# ============================================================
# STEP 4: RUN ALL 125 PROMPTS
# ============================================================

all_results = {}

for idx, (label, prompt) in enumerate(PROMPT_LIBRARY.items()):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(PROMPT_LIBRARY)}] RECORDING: '{label}'")
    print(f"  Prompt: \"{prompt}\"")
    print(f"{'='*60}")
    
    snapshots = run_total_resonance_loop(
        model, prompt,
        layer_start=LAYER_START,
        layer_end=LAYER_END,
        max_iter=MAX_ITERATIONS,
        schedule=ITERATION_SCHEDULE
    )
    all_results[label] = snapshots
    
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    print(f"  Terminal token: '{terminal}'")

print(f"\n{'='*60}")
print(f"ALL {len(all_results)} RECORDINGS COMPLETE.")

---
## 5. Analysis

### 5a. Hypothesis Assessment — Predictions vs Actuals

In [18]:
# ============================================================
# VIS 5a: HYPOTHESIS ASSESSMENT — Predictions vs Actuals
# ============================================================

md = "# Stage 1 Results: Hypothesis Assessment\n\n"
md += "| Prompt | Category | Predicted | Actual Terminal | Match? |\n"
md += "|:---|:---|:---|:---|:---|\n"

basin_counts = {}
category_basins = {}
mismatches = []

for label, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    predicted_basin, confidence = PREDICTIONS[label]
    category = CATEGORY_MAP[label]
    
    # Classify actual basin
    # AFTER (exact match)
    if terminal == 'prolet':
        actual_basin = 'prolet'
    elif terminal == 'Divine':
        actual_basin = 'Divine'
    else:
        actual_basin = f'OTHER:{terminal}'
    
    basin_counts[actual_basin] = basin_counts.get(actual_basin, 0) + 1
    
    if category not in category_basins:
        category_basins[category] = []
    category_basins[category].append((label, actual_basin, terminal))
    
    match = 'y' if predicted_basin.lower() in actual_basin.lower() else 'n'
    if match == 'n' and predicted_basin != 'unknown':
        mismatches.append((label, predicted_basin, actual_basin))
    
    md += f"| {label} | {category} | `{predicted_basin}` ({confidence}) | `{terminal}` -> **{actual_basin}** | {match} |\n"

md += "\n---\n\n"
md += "## Basin Summary\n\n"
md += "| Basin | Count | % |\n"
md += "|:---|:---|:---|\n"
total = len(all_results)
for basin, count in sorted(basin_counts.items(), key=lambda x: -x[1]):
    md += f"| **{basin}** | {count} | {count/total*100:.1f}% |\n"

md += "\n---\n\n"
md += "## Category Breakdown\n\n"
for cat, entries in category_basins.items():
    md += f"### {cat} ({len(entries)} prompts)\n"
    cat_basins = {}
    for label, basin, tok in entries:
        cat_basins[basin] = cat_basins.get(basin, 0) + 1
    for b, c in sorted(cat_basins.items(), key=lambda x: -x[1]):
        md += f"- {b}: {c}/{len(entries)}\n"
    md += "\n"

if mismatches:
    md += "## Prediction Mismatches\n\n"
    for label, pred, actual in mismatches:
        md += f"- **{label}**: predicted `{pred}`, got `{actual}`\n"

# Save and display
with open(os.path.join(OUTPUT_DIR, 'hypothesis_assessment.md'), 'w') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/hypothesis_assessment.md")
display(Markdown(md))

[SAVED] output_stage1/hypothesis_assessment.md


# Stage 1 Results: Hypothesis Assessment

| Prompt | Category | Predicted | Actual Terminal | Match? |
|:---|:---|:---|:---|:---|
| A01_physics | Complex | `prolet` (high) | `prolet` -> **prolet** | y |
| A02_medical | Complex | `prolet` (high) | `prolet` -> **prolet** | y |
| A03_neuro | Complex | `prolet` (high) | `Anarch` -> **OTHER:Anarch** | n |
| A04_climate | Complex | `prolet` (high) | `prolet` -> **prolet** | y |
| A05_evolution | Complex | `prolet` (high) | `Anarch` -> **OTHER:Anarch** | n |
| A06_epistemology | Complex | `prolet` (high) | `Anarch` -> **OTHER:Anarch** | n |
| A07_sociology | Complex | `prolet` (high) | `Anarch` -> **OTHER:Anarch** | n |
| A08_linguistics | Complex | `prolet` (high) | `Divine` -> **Divine** | n |
| A09_code | Complex | `prolet` (high) | `Anarch` -> **OTHER:Anarch** | n |
| A10_sql | Complex | `prolet` (high) | `prolet` -> **prolet** | y |
| A11_ml | Complex | `prolet` (high) | `Anarch` -> **OTHER:Anarch** | n |
| A12_systems | Complex | `prolet` (high) | `prolet` -> **prolet** | y |
| A13_networking | Complex | `prolet` (high) | `prolet` -> **prolet** | y |
| A14_kant | Complex | `prolet` (high) | `Divine` -> **Divine** | n |
| A15_sartre | Complex | `prolet` (high) | `Divine` -> **Divine** | n |
| A16_wittgenstein | Complex | `prolet` (high) | `prolet` -> **prolet** | y |
| A17_marx | Complex | `prolet` (high) | `Divine` -> **Divine** | n |
| A18_gothic | Complex | `prolet` (high) | `Anarch` -> **OTHER:Anarch** | n |
| A19_romantic | Complex | `prolet` (high) | `Anarch` -> **OTHER:Anarch** | n |
| A20_modernist | Complex | `prolet` (high) | `prolet` -> **prolet** | y |
| A21_dickens | Complex | `prolet` (high) | `Divine` -> **Divine** | n |
| A22_legal | Complex | `prolet` (high) | `Anarch` -> **OTHER:Anarch** | n |
| A23_contract | Complex | `prolet` (high) | `prolet` -> **prolet** | y |
| A24_patent | Complex | `prolet` (high) | `Anarch` -> **OTHER:Anarch** | n |
| A25_academic_abs | Complex | `prolet` (high) | `Divine` -> **Divine** | n |
| B01_napoleon | Narrative | `prolet` (medium) | `prolet` -> **prolet** | y |
| B02_wwi | Narrative | `prolet` (medium) | `prolet` -> **prolet** | y |
| B03_moon | Narrative | `prolet` (medium) | `Divine` -> **Divine** | n |
| B04_rome | Narrative | `prolet` (medium) | `prolet` -> **prolet** | y |
| B05_mlk | Narrative | `prolet` (medium) | `till` -> **OTHER:till** | n |
| B06_sources | Narrative | `prolet` (medium) | `Anarch` -> **OTHER:Anarch** | n |
| B07_breaking | Narrative | `prolet` (medium) | `Anarch` -> **OTHER:Anarch** | n |
| B08_editorial | Narrative | `prolet` (medium) | `Anarch` -> **OTHER:Anarch** | n |
| B09_sports | Narrative | `prolet` (medium) | `Anarch` -> **OTHER:Anarch** | n |
| B10_weather | Narrative | `prolet` (medium) | `prolet` -> **prolet** | y |
| B11_alone | Narrative | `prolet` (medium) | `till` -> **OTHER:till** | n |
| B12_fear | Narrative | `prolet` (medium) | `Divine` -> **Divine** | n |
| B13_joy | Narrative | `prolet` (medium) | `Divine` -> **Divine** | n |
| B14_anger | Narrative | `prolet` (medium) | `Anarch` -> **OTHER:Anarch** | n |
| B15_casual | Narrative | `prolet` (medium) | `till` -> **OTHER:till** | n |
| B16_gossip | Narrative | `prolet` (medium) | `Divine` -> **Divine** | n |
| B17_argument | Narrative | `prolet` (medium) | `Divine` -> **Divine** | n |
| B18_advice | Narrative | `prolet` (medium) | `till` -> **OTHER:till** | n |
| B19_question | Narrative | `prolet` (medium) | `till` -> **OTHER:till** | n |
| B20_reddit | Narrative | `prolet` (medium) | `Anarch` -> **OTHER:Anarch** | n |
| D01_water | Chemical | `unknown` (none) | `prolet` -> **prolet** | n |
| D02_periodic | Chemical | `unknown` (none) | `till` -> **OTHER:till** | n |
| D03_organic | Chemical | `unknown` (none) | `till` -> **OTHER:till** | n |
| D04_equation | Chemical | `unknown` (none) | `Anarch` -> **OTHER:Anarch** | n |
| D05_amino | Chemical | `unknown` (none) | `prolet` -> **prolet** | n |
| D06_physics_eq | Chemical | `unknown` (none) | `prolet` -> **prolet** | n |
| D07_dna | Chemical | `unknown` (none) | `till` -> **OTHER:till** | n |
| D08_math | Chemical | `unknown` (none) | `solidarity` -> **OTHER:solidarity** | n |
| D09_units | Chemical | `unknown` (none) | `Divine` -> **Divine** | n |
| D10_isotopes | Chemical | `unknown` (none) | `prolet` -> **prolet** | n |
| E01_politics | Acronyms | `unknown` (none) | `prolet` -> **prolet** | n |
| E02_tech | Acronyms | `unknown` (none) | `Divine` -> **Divine** | n |
| E03_orgs | Acronyms | `unknown` (none) | `Divine` -> **Divine** | n |
| E04_internet | Acronyms | `unknown` (none) | `Divine` -> **Divine** | n |
| E05_finance | Acronyms | `unknown` (none) | `Anarch` -> **OTHER:Anarch** | n |
| E06_medical | Acronyms | `unknown` (none) | `Divine` -> **Divine** | n |
| E07_military | Acronyms | `unknown` (none) | `prolet` -> **prolet** | n |
| E08_academic | Acronyms | `unknown` (none) | `Anarch` -> **OTHER:Anarch** | n |
| E09_mixed | Acronyms | `unknown` (none) | `Divine` -> **Divine** | n |
| E10_crypto | Acronyms | `unknown` (none) | `prolet` -> **prolet** | n |
| C01_jack_jill | Simple | `Divine` (low) | `Anarch` -> **OTHER:Anarch** | n |
| C02_king_cole | Simple | `Divine` (low) | `prolet` -> **prolet** | n |
| C03_mary_lamb | Simple | `Divine` (low) | `prolet` -> **prolet** | n |
| C04_humpty | Simple | `Divine` (low) | `prolet` -> **prolet** | n |
| C05_twinkle | Simple | `Divine` (low) | `Divine` -> **Divine** | y |
| C06_dog | Simple | `Divine` (low) | `prolet` -> **prolet** | n |
| C07_cat_mat | Simple | `Divine` (low) | `Divine` -> **Divine** | y |
| C08_boy_girl | Simple | `Divine` (low) | `prolet` -> **prolet** | n |
| C09_run | Simple | `Divine` (low) | `prolet` -> **prolet** | n |
| C10_spot | Simple | `Divine` (low) | `till` -> **OTHER:till** | n |
| C11_genesis | Simple | `Divine` (low) | `Anarch` -> **OTHER:Anarch** | n |
| C12_beatitudes | Simple | `Divine` (low) | `prolet` -> **prolet** | n |
| C13_psalm | Simple | `Divine` (low) | `Divine` -> **Divine** | y |
| C14_commandment | Simple | `Divine` (low) | `prolet` -> **prolet** | n |
| C15_fox_hen | Simple | `Divine` (low) | `prolet` -> **prolet** | n |
| C16_ant_dove | Simple | `Divine` (low) | `Divine` -> **Divine** | y |
| C17_tortoise | Simple | `Divine` (low) | `prolet` -> **prolet** | n |
| C18_wolf | Simple | `Divine` (low) | `Divine` -> **Divine** | y |
| C19_lion_mouse | Simple | `Divine` (low) | `prolet` -> **prolet** | n |
| C20_crow | Simple | `Divine` (low) | `Anarch` -> **OTHER:Anarch** | n |
| F01_anger | Vulgarity | `unknown` (none) | `Divine` -> **Divine** | n |
| F02_insult | Vulgarity | `unknown` (none) | `Divine` -> **Divine** | n |
| F03_frustration | Vulgarity | `unknown` (none) | `Divine` -> **Divine** | n |
| F04_argument | Vulgarity | `unknown` (none) | `till` -> **OTHER:till** | n |
| F05_rant | Vulgarity | `unknown` (none) | `Divine` -> **Divine** | n |
| F06_dismissal | Vulgarity | `unknown` (none) | `till` -> **OTHER:till** | n |
| F07_shock | Vulgarity | `unknown` (none) | `Divine` -> **Divine** | n |
| F08_mild | Vulgarity | `unknown` (none) | `till` -> **OTHER:till** | n |
| F09_slur_adjacent | Vulgarity | `unknown` (none) | `Divine` -> **Divine** | n |
| F10_exasperation | Vulgarity | `unknown` (none) | `Anarch` -> **OTHER:Anarch** | n |
| G01_punctuation | Wild | `unknown` (none) | `till` -> **OTHER:till** | n |
| G02_brackets | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G03_counting | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G04_fibonacci | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G05_primes | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G06_binary | Wild | `unknown` (none) | `till` -> **OTHER:till** | n |
| G07_the | Wild | `unknown` (none) | `Anarch` -> **OTHER:Anarch** | n |
| G08_period | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G09_space | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G10_newline | Wild | `unknown` (none) | `solidarity` -> **OTHER:solidarity** | n |
| G11_aaa | Wild | `unknown` (none) | `till` -> **OTHER:till** | n |
| G12_the_the | Wild | `unknown` (none) | `till` -> **OTHER:till** | n |
| G13_buffalo | Wild | `unknown` (none) | `Divine` -> **Divine** | n |
| G14_nursery_acad | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G15_bible_code | Wild | `unknown` (none) | `Divine` -> **Divine** | n |
| G16_nursery_vulgar | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G17_formal_slang | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G18_french | Wild | `unknown` (none) | `till` -> **OTHER:till** | n |
| G19_german | Wild | `unknown` (none) | `Anarch` -> **OTHER:Anarch** | n |
| G20_spanish | Wild | `unknown` (none) | `Divine` -> **Divine** | n |
| G21_latin | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G22_japanese_rom | Wild | `unknown` (none) | `till` -> **OTHER:till** | n |
| G23_emoji | Wild | `unknown` (none) | `Divine` -> **Divine** | n |
| G24_beatles | Wild | `unknown` (none) | `Anarch` -> **OTHER:Anarch** | n |
| G25_rickroll | Wild | `unknown` (none) | `till` -> **OTHER:till** | n |
| G26_bohemian | Wild | `unknown` (none) | `Divine` -> **Divine** | n |
| G27_ignore | Wild | `unknown` (none) | `Divine` -> **Divine** | n |
| G28_system | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G29_palindrome | Wild | `unknown` (none) | `prolet` -> **prolet** | n |
| G30_alphabet | Wild | `unknown` (none) | `prolet` -> **prolet** | n |

---

## Basin Summary

| Basin | Count | % |
|:---|:---|:---|
| **prolet** | 44 | 35.2% |
| **Divine** | 34 | 27.2% |
| **OTHER:Anarch** | 26 | 20.8% |
| **OTHER:till** | 19 | 15.2% |
| **OTHER:solidarity** | 2 | 1.6% |

---

## Category Breakdown

### Complex (25 prompts)
- OTHER:Anarch: 10/25
- prolet: 9/25
- Divine: 6/25

### Narrative (20 prompts)
- OTHER:Anarch: 6/20
- Divine: 5/20
- OTHER:till: 5/20
- prolet: 4/20

### Chemical (10 prompts)
- prolet: 4/10
- OTHER:till: 3/10
- OTHER:Anarch: 1/10
- OTHER:solidarity: 1/10
- Divine: 1/10

### Acronyms (10 prompts)
- Divine: 5/10
- prolet: 3/10
- OTHER:Anarch: 2/10

### Simple (20 prompts)
- prolet: 11/20
- Divine: 5/20
- OTHER:Anarch: 3/20
- OTHER:till: 1/20

### Vulgarity (10 prompts)
- Divine: 6/10
- OTHER:till: 3/10
- OTHER:Anarch: 1/10

### Wild (30 prompts)
- prolet: 13/30
- OTHER:till: 7/30
- Divine: 6/30
- OTHER:Anarch: 3/30
- OTHER:solidarity: 1/30

## Prediction Mismatches

- **A03_neuro**: predicted `prolet`, got `OTHER:Anarch`
- **A05_evolution**: predicted `prolet`, got `OTHER:Anarch`
- **A06_epistemology**: predicted `prolet`, got `OTHER:Anarch`
- **A07_sociology**: predicted `prolet`, got `OTHER:Anarch`
- **A08_linguistics**: predicted `prolet`, got `Divine`
- **A09_code**: predicted `prolet`, got `OTHER:Anarch`
- **A11_ml**: predicted `prolet`, got `OTHER:Anarch`
- **A14_kant**: predicted `prolet`, got `Divine`
- **A15_sartre**: predicted `prolet`, got `Divine`
- **A17_marx**: predicted `prolet`, got `Divine`
- **A18_gothic**: predicted `prolet`, got `OTHER:Anarch`
- **A19_romantic**: predicted `prolet`, got `OTHER:Anarch`
- **A21_dickens**: predicted `prolet`, got `Divine`
- **A22_legal**: predicted `prolet`, got `OTHER:Anarch`
- **A24_patent**: predicted `prolet`, got `OTHER:Anarch`
- **A25_academic_abs**: predicted `prolet`, got `Divine`
- **B03_moon**: predicted `prolet`, got `Divine`
- **B05_mlk**: predicted `prolet`, got `OTHER:till`
- **B06_sources**: predicted `prolet`, got `OTHER:Anarch`
- **B07_breaking**: predicted `prolet`, got `OTHER:Anarch`
- **B08_editorial**: predicted `prolet`, got `OTHER:Anarch`
- **B09_sports**: predicted `prolet`, got `OTHER:Anarch`
- **B11_alone**: predicted `prolet`, got `OTHER:till`
- **B12_fear**: predicted `prolet`, got `Divine`
- **B13_joy**: predicted `prolet`, got `Divine`
- **B14_anger**: predicted `prolet`, got `OTHER:Anarch`
- **B15_casual**: predicted `prolet`, got `OTHER:till`
- **B16_gossip**: predicted `prolet`, got `Divine`
- **B17_argument**: predicted `prolet`, got `Divine`
- **B18_advice**: predicted `prolet`, got `OTHER:till`
- **B19_question**: predicted `prolet`, got `OTHER:till`
- **B20_reddit**: predicted `prolet`, got `OTHER:Anarch`
- **C01_jack_jill**: predicted `Divine`, got `OTHER:Anarch`
- **C02_king_cole**: predicted `Divine`, got `prolet`
- **C03_mary_lamb**: predicted `Divine`, got `prolet`
- **C04_humpty**: predicted `Divine`, got `prolet`
- **C06_dog**: predicted `Divine`, got `prolet`
- **C08_boy_girl**: predicted `Divine`, got `prolet`
- **C09_run**: predicted `Divine`, got `prolet`
- **C10_spot**: predicted `Divine`, got `OTHER:till`
- **C11_genesis**: predicted `Divine`, got `OTHER:Anarch`
- **C12_beatitudes**: predicted `Divine`, got `prolet`
- **C14_commandment**: predicted `Divine`, got `prolet`
- **C15_fox_hen**: predicted `Divine`, got `prolet`
- **C17_tortoise**: predicted `Divine`, got `prolet`
- **C19_lion_mouse**: predicted `Divine`, got `prolet`
- **C20_crow**: predicted `Divine`, got `OTHER:Anarch`


### 5b. Cross-Prompt Convergence Matrix

In [19]:
# ============================================================
# VIS 5b: CROSS-PROMPT CONVERGENCE MATRIX
# ============================================================

labels = list(all_results.keys())
n = len(labels)
sim_matrix = np.zeros((n, n))

final_vectors = []
for label in labels:
    final_vec = all_results[label][-1]["mean_vector"]
    final_vectors.append(final_vec)

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = torch.nn.functional.cosine_similarity(
            final_vectors[i].unsqueeze(0).float(),
            final_vectors[j].unsqueeze(0).float()
        ).item()

fig_sim = px.imshow(
    sim_matrix,
    x=labels, y=labels,
    color_continuous_scale="Viridis",
    title="Stage 1: Cross-Prompt Convergence (125 Prompts)",
    aspect="auto",
)
fig_sim.update_layout(template="plotly_dark", height=900, width=1200)
fig_sim.show()
fig_sim.write_image(os.path.join(OUTPUT_DIR, 'convergence_matrix.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/convergence_matrix.png")

off_diag = sim_matrix[np.triu_indices(n, k=1)]
print(f"\nMean cross-prompt similarity: {off_diag.mean():.4f}")
print(f"Min:  {off_diag.min():.4f}")
print(f"Max:  {off_diag.max():.4f}")

[SAVED] output_stage1/convergence_matrix.png

Mean cross-prompt similarity: 0.8649
Min:  0.6689
Max:  1.0000


### 5c. Dissolution Pathway Analysis

In [24]:
# ============================================================
# VIS 5c: DISSOLUTION PATHWAYS — Per Category
# ============================================================

md = "# Dissolution Pathways — Last-Token Top Prediction\n\n"

for cat_name, cat_dict in [
    ("Complex", COMPLEX), ("Narrative", NARRATIVE),
    ("Simple", SIMPLE), ("Chemical", CHEMICAL),
    ("Acronyms", ACRONYMS), ("Vulgarity", VULGARITY),
    ("Wild", WILD)
]:
    cat_labels = [k for k in cat_dict.keys() if k in all_results]
    if not cat_labels:
        continue
    
    md += f"## {cat_name} ({len(cat_labels)} prompts)\n\n"
    md += "| Iter | " + " | ".join(cat_labels[:10]) + " |\n"
    md += "| :--- | " + " | ".join([":---"] * min(len(cat_labels), 10)) + " |\n"
    
    for idx, iteration in enumerate(ITERATION_SCHEDULE):
        row = f"| **{iteration}** |"
        for label in cat_labels[:10]:
            snapshots = all_results[label]
            if idx < len(snapshots):
                tok = snapshots[idx]['top_tokens'][0][0]
                clean_t = tok.replace('\n', '↵').replace('`', "'").strip()
                row += f" `{clean_t}` |"
            else:
                row += " — |"
        md += row + "\n"
    md += "\n"

with open(os.path.join(OUTPUT_DIR, 'dissolution_pathways.md'), 'w', encoding='utf-8') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/dissolution_pathways.md")
display(Markdown(md))

[SAVED] output_stage1/dissolution_pathways.md


# Dissolution Pathways — Last-Token Top Prediction

## Complex (25 prompts)

| Iter | A01_physics | A02_medical | A03_neuro | A04_climate | A05_evolution | A06_epistemology | A07_sociology | A08_linguistics | A09_code | A10_sql |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `the` | `that` | `the` | `global` | `that` | `the` | `and` | `the` | `object` | `'` |
| **2** | `-` | `the` | `the` | `↵` | `the` | `the` | `the` | `the` | `↵` | `↵` |
| **3** | `,` | `the` | `the` | `.` | `the` | `the` | `the` | `the` | `↵` | `↵` |
| **5** | `Canad` | `Canad` | `fem` | `.` | `Canad` | `fem` | `fem` | `fem` | `Ada` | `NP` |
| **10** | `capit` | `capit` | `capit` | `capit` | `Ag` | `capit` | `Ag` | `capit` | `Ag` | `Ag` |
| **20** | `injustice` | `injustice` | `.` | `.` | `Zero` | `.` | `.` | `Zero` | `.` | `.` |
| **50** | `Rousse` | `Rousse` | `Rousse` | `Rousse` | `―` | `Rousse` | `abstract` | `―` | `Rousse` | `Rousse` |
| **100** | `prolet` | `prolet` | `Anarch` | `prolet` | `Anarch` | `Anarch` | `Anarch` | `Divine` | `Anarch` | `prolet` |

## Narrative (20 prompts)

| Iter | B01_napoleon | B02_wwi | B03_moon | B04_rome | B05_mlk | B06_sources | B07_breaking | B08_editorial | B09_sports | B10_weather |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `his` | `,` | `for` | `fifty` | `dream` | `team` | `FBI` | `election` | `,` | `the` |
| **2** | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `the` | `the` |
| **3** | `the` | `↵` | `↵` | `the` | `↵` | `↵` | `↵` | `↵` | `the` | `the` |
| **5** | `Fem` | `↵` | `Canad` | `Fem` | `↵` | `↵` | `Canad` | `Fem` | `↵` | `Canad` |
| **10** | `capit` | `Equ` | `Ag` | `swap` | `Ag` | `Ag` | `FT` | `Ag` | `capit` | `Ag` |
| **20** | `injustice` | `trade` | `Zero` | `.` | `Zero` | `Zero` | `.` | `.` | `.` | `.` |
| **50** | `Rousse` | `prolet` | `Divine` | `Rousse` | `Nan` | `solidarity` | `Rousse` | `Rousse` | `prolet` | `Rousse` |
| **100** | `prolet` | `prolet` | `Divine` | `prolet` | `till` | `Anarch` | `Anarch` | `Anarch` | `Anarch` | `prolet` |

## Simple (20 prompts)

| Iter | C01_jack_jill | C02_king_cole | C03_mary_lamb | C04_humpty | C05_twinkle | C06_dog | C07_cat_mat | C08_boy_girl | C09_run | C10_spot |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `the` | `who` | `cooked` | `sat` | `it` | `and` | `watch` | `hospital` | `will` | `run` |
| **2** | `the` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` |
| **3** | `the` | `↵` | `the` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` |
| **5** | `Canad` | `Canad` | `Canad` | `Canad` | `Canad` | `Canad` | `↵` | `Canad` | `NP` | `↵` |
| **10** | `Ag` | `Ag` | `swap` | `capit` | `Ag` | `Ag` | `Ag` | `capit` | `equivalent` | `Ag` |
| **20** | `Zero` | `Difference` | `.` | `injustice` | `Zero` | `.` | `Zero` | `.` | `equival` | `Zero` |
| **50** | `instant` | `Rousse` | `Rousse` | `Rousse` | `Divine` | `Rousse` | `―` | `Rousse` | `Rousse` | `solidarity` |
| **100** | `Anarch` | `prolet` | `prolet` | `prolet` | `Divine` | `prolet` | `Divine` | `prolet` | `prolet` | `till` |

## Chemical (10 prompts)

| Iter | D01_water | D02_periodic | D03_organic | D04_equation | D05_amino | D06_physics_eq | D07_dna | D08_math | D09_units | D10_isotopes |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `NH` | `1` | `C` | `1` | `↵` | `=` | `G` | `the` | `r` | `C` |
| **2** | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` |
| **3** | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` |
| **5** | `↵` | `↵` | `AF` | `↵` | `Canad` | `minus` | `AF` | `0` | `ash` | `↵` |
| **10** | `FT` | `Ag` | `Ag` | `Ag` | `capit` | `capit` | `Ag` | `Ag` | `Ag` | `Ag` |
| **20** | `injustice` | `Same` | `Zero` | `Zero` | `.` | `injustice` | `Same` | `Zero` | `Zero` | `.` |
| **50** | `Rousse` | `solidarity` | `solidarity` | `abstract` | `Rousse` | `Rousse` | `solidarity` | `―` | `Divine` | `Rousse` |
| **100** | `prolet` | `till` | `till` | `Anarch` | `prolet` | `prolet` | `till` | `solidarity` | `Divine` | `prolet` |

## Acronyms (10 prompts)

| Iter | E01_politics | E02_tech | E03_orgs | E04_internet | E05_finance | E06_medical | E07_military | E08_academic | E09_mixed | E10_crypto |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `G` | `SSH` | `CDC` | `↵` | `E` | `OR` | `I` | `↵` | `T` | `DA` |
| **2** | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` |
| **3** | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` |
| **5** | `Ag` | `API` | `Ag` | `WHAT` | `↵` | `NP` | `Canad` | `Canad` | `Canad` | `↵` |
| **10** | `Ag` | `Ag` | `Ag` | `Ag` | `Ag` | `Ag` | `FT` | `Ag` | `Ag` | `Ag` |
| **20** | `injustice` | `Zero` | `Zero` | `Zero` | `.` | `Zero` | `.` | `Zero` | `Zero` | `injustice` |
| **50** | `Rousse` | `Divine` | `Divine` | `Divine` | `Rousse` | `Divine` | `Rousse` | `Rousse` | `Divine` | `Rousse` |
| **100** | `prolet` | `Divine` | `Divine` | `Divine` | `Anarch` | `Divine` | `prolet` | `Anarch` | `Divine` | `prolet` |

## Vulgarity (10 prompts)

| Iter | F01_anger | F02_insult | F03_frustration | F04_argument | F05_rant | F06_dismissal | F07_shock | F08_mild | F09_slur_adjacent | F10_exasperation |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `?` | `about` | `have` | `fucking` | `should` | `about` | `to` | `thing` | `ever` | `you` |
| **2** | `↵` | `↵` | `.` | `↵` | `is` | `↵` | `↵` | `↵` | `↵` | `↵` |
| **3** | `↵` | `↵` | `.` | `↵` | `.` | `↵` | `↵` | `↵` | `↵` | `↵` |
| **5** | `↵` | `.` | `fem` | `ash` | `.` | `.` | `cows` | `.` | `↵` | `Either` |
| **10** | `Ag` | `Ag` | `Ag` | `Ag` | `Ag` | `Ag` | `ash` | `ash` | `Ag` | `Ag` |
| **20** | `Zero` | `Zero` | `Zero` | `Zero` | `Zero` | `Zero` | `Zero` | `Zero` | `Zero` | `Zero` |
| **50** | `Divine` | `Divine` | `Divine` | `solidarity` | `Divine` | `solidarity` | `Divine` | `solidarity` | `Divine` | `―` |
| **100** | `Divine` | `Divine` | `Divine` | `till` | `Divine` | `till` | `Divine` | `till` | `Divine` | `Anarch` |

## Wild (30 prompts)

| Iter | G01_punctuation | G02_brackets | G03_counting | G04_fibonacci | G05_primes | G06_binary | G07_the | G08_period | G09_space | G10_newline |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `;` | `|` | `10` | `↵` | `30` | `01` | `first` | `↵` | `` | `The` |
| **2** | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `` | `↵` |
| **3** | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `The` |
| **5** | `↵` | `↵` | `↵` | `↵` | `↵` | `↵` | `Canad` | `ash` | `ash` | `Posted` |
| **10** | `Ag` | `Ag` | `capit` | `Ag` | `Ag` | `Ag` | `Ag` | `FT` | `FT` | `Ag` |
| **20** | `Zero` | `.` | `injustice` | `injustice` | `injustice` | `Same` | `Zero` | `injustice` | `injustice` | `N` |
| **50** | `solidarity` | `Rousse` | `Rousse` | `Rousse` | `Rousse` | `solidarity` | `instant` | `Rousse` | `Rousse` | `solidarity` |
| **100** | `till` | `prolet` | `prolet` | `prolet` | `prolet` | `till` | `Anarch` | `prolet` | `prolet` | `solidarity` |



### 5d. Sentence Dissolution Tables — Full Position Reconstruction

In [23]:
# ============================================================
# VIS 5d: SENTENCE DISSOLUTION TABLES
# ============================================================

md = "# Full Sentence Dissolution — All 125 Prompts\n\n"

for label in PROMPT_LIBRARY.keys():
    if label not in all_results:
        continue
    snapshots = all_results[label]
    predicted, conf = PREDICTIONS[label]
    category = CATEGORY_MAP[label]
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    
    md += f"### {label} [{category}] -> `{terminal}` (predicted: `{predicted}`)\n"
    md += f"*\"{PROMPT_LIBRARY[label]}\"*\n\n"
    md += "| Iter | Reconstructed Output |\n"
    md += "|:---|:---|\n"
    for s in snapshots:
        tokens = s['all_position_tokens']
        clean = [t.replace('\n', '↵').replace('|', '\\|') for t in tokens]
        sentence = ' '.join(clean)
        md += f"| {s['iteration']} | {sentence} |\n"
    md += "\n"

with open(os.path.join(OUTPUT_DIR, 'dissolution_sentences.md'), 'w', encoding='utf-8') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/dissolution_sentences.md")
print(f"Full dissolution tables: {len(all_results)} prompts written.")

[SAVED] output_stage1/dissolution_sentences.md
Full dissolution tables: 125 prompts written.


### 5e. 3D PCA Trajectories — All Prompts

In [ ]:
# ============================================================
# VIS 5e: 3D PCA TRAJECTORIES
# ============================================================
from sklearn.decomposition import PCA
import pandas as pd

all_vecs = []
labels_list = []
cats_list = []
iters_list = []
text_list = []

for label, snapshots in all_results.items():
    for s in snapshots:
        all_vecs.append(s["mean_vector"].detach().cpu().numpy())
        labels_list.append(label)
        cats_list.append(CATEGORY_MAP.get(label, 'Unknown'))
        iters_list.append(s["iteration"])
        top_tok = s['top_tokens'][0][0].replace('\n', '↵').strip()
        text_list.append(f"Iter {s['iteration']}: {top_tok}")

all_vecs = np.array(all_vecs)
pca = PCA(n_components=3)
vecs_3d = pca.fit_transform(all_vecs)

df = pd.DataFrame({
    'x': vecs_3d[:, 0],
    'y': vecs_3d[:, 1],
    'z': vecs_3d[:, 2],
    'Prompt': labels_list,
    'Category': cats_list,
    'Iteration': iters_list,
    'Top_Token': text_list
})

# Color by category for readability
fig_topo = px.line_3d(
    df, x='x', y='y', z='z',
    color='Category',
    hover_name='Top_Token',
    markers=True,
    title=f"Stage 1: Attractor Landscape — {len(all_results)} Prompt Trajectories<br>"
          f"<sup>(Explained Variance: {sum(pca.explained_variance_ratio_)*100:.1f}%)</sup>"
)
fig_topo.update_traces(marker=dict(size=3), line=dict(width=2))
fig_topo.update_layout(
    template="plotly_dark",
    height=900,
    width=1200,
    scene=dict(
        xaxis_title="PC 1",
        yaxis_title="PC 2",
        zaxis_title="PC 3",
    )
)
fig_topo.show()
fig_topo.write_image(os.path.join(OUTPUT_DIR, 'topology_3d.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/topology_3d.png")

### 5f. Basin Distribution Chart

In [ ]:
# ============================================================
# VIS 5f: BASIN DISTRIBUTION BAR CHART
# ============================================================

basin_data = []
for label, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    category = CATEGORY_MAP.get(label, 'Unknown')
    if terminal == 'prolet':
        basin = 'prolet'
    elif terminal == 'Divine':
        basin = 'Divine'
    else:
        basin = terminal
    basin_data.append({'Prompt': label, 'Category': category, 'Basin': basin})

basin_df = pd.DataFrame(basin_data)
basin_summary = basin_df.groupby(['Category', 'Basin']).size().reset_index(name='Count')

fig_basin = px.bar(
    basin_summary, x='Category', y='Count', color='Basin',
    title=f"Stage 1: Basin Distribution by Category ({len(all_results)} prompts)",
    barmode='stack'
)
fig_basin.update_layout(template="plotly_dark", height=500, width=900)
fig_basin.show()
fig_basin.write_image(os.path.join(OUTPUT_DIR, 'basin_distribution.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/basin_distribution.png")

In [ ]:
# ============================================================
# STEP 6: SAVE RAW DATA
# ============================================================

save_data = {}
for label, snapshots in all_results.items():
    save_data[label] = {
        "iterations": [s["iteration"] for s in snapshots],
        "last_vectors": torch.stack([s["last_vector"] for s in snapshots]),
        "mean_vectors": torch.stack([s["mean_vector"] for s in snapshots]),
        "last_norms": [s["last_norm"] for s in snapshots],
        "mean_norms": [s["mean_norm"] for s in snapshots],
        "cosine_sims_last": [s["cosine_sim_last"] for s in snapshots],
        "cosine_sims_mean": [s["cosine_sim_mean"] for s in snapshots],
        "position_similarity": [s["position_similarity"] for s in snapshots],
        "top_tokens": [s["top_tokens"] for s in snapshots],
        "all_position_tokens": [s["all_position_tokens"] for s in snapshots],
    }

torch.save(save_data, os.path.join(OUTPUT_DIR, 'stage1_results.pt'))
print(f"[SAVED] {OUTPUT_DIR}/stage1_results.pt")

config = {
    "schedule": ITERATION_SCHEDULE,
    "layer_start": LAYER_START,
    "layer_end": LAYER_END,
    "prompt_count": len(PROMPT_LIBRARY),
    "model": "gpt2-small",
    "mode": "stage1_attractor_dominance",
}
torch.save(config, os.path.join(OUTPUT_DIR, 'stage1_config.pt'))
print(f"[SAVED] {OUTPUT_DIR}/stage1_config.pt")
print(f"\ny All artifacts saved to {os.path.abspath(OUTPUT_DIR)}")